In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
forbidden_topics = {
    "cheating": ["답지", "정답 알려줘", "숙제 대신", "써줘", "베끼기"],
    "distraction": ["롤", "게임", "유튜브", "아이돌", "웹툰", "웃긴"],
    "harmful": ["담배", "술", "폭력", "싸움", "바보"]
}

In [4]:
from langchain.agents.middleware import before_agent

@before_agent(can_jump_to=["end"])
def education_guardrail(state, runtime):
    if not state["messages"]:
        return None
    
    last_msg = state["messages"][-1]
    if last_msg.type != "human":
        return None
    
    user_text = last_msg.content

    for kw in forbidden_topics["cheating"]:
        if kw in user_text:
            return {
                "messages": [
                        {
                            "role": "assistant",
                            "content": "🙊스스로 고민해봐야 실력이 늘어요! 정답을 바로 알려드리는 대신, 힌트를 들릴까요?"
                        }
                    ],
                    "jump_to": "end"
            }
        
    for kw in forbidden_topics["distraction"]:
        if kw in user_text:
            return {
                "messages": [
                        {
                            "role": "assistant",
                            "content": "⏰지금은 공부에 집중할 시간이에요! 딴짓은 쉬는 시간에 하고, 지금은 풀고 있는 문제에 집중해볼까요?"
                        }
                    ],
                    "jump_to": "end"
            }
        
    for kw in forbidden_topics["harmful"]:
        if kw in user_text:
            return {
                "messages": [
                        {
                            "role": "assistant",
                            "content": "⚠️부적절한 대화 주제입니다. 바르고 고운 말을 사용해주세요."
                        }
                    ],
                    "jump_to": "end"
            }
        
    return None

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    middleware=[education_guardrail],
)

In [6]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "랭체인으로 할 수 있는 일은 뭘까?"}]
})

In [7]:
result

{'messages': [HumanMessage(content='랭체인으로 할 수 있는 일은 뭘까?', additional_kwargs={}, response_metadata={}, id='98457ba4-f35a-45dc-913e-bd0b095930e8'),
  AIMessage(content=[{'type': 'text', 'text': '**랭체인(LangChain)**은 거대 언어 모델(LLM, 예: GPT-4, Claude, Llama 등)을 활용해 **복잡한 애플리케이션을 쉽고 효율적으로 개발할 수 있게 도와주는 프레임워크**입니다.\n\n간단히 말해, LLM에게 \'뇌\'가 있다면 랭체인은 그 뇌에 **\'손과 발, 기억력, 그리고 외부 정보망\'**을 달아주는 도구라고 생각하면 됩니다. 랭체인으로 할 수 있는 대표적인 일들은 다음과 같습니다.\n\n---\n\n### 1. RAG (Retrieval-Augmented Generation, 검색 증강 생성)\nLLM이 학습하지 않은 최신 정보나 기업 내부의 비공개 데이터를 참조하게 만드는 기술입니다.\n*   **활용 예시:** PDF, Word, 웹사이트 데이터를 랭체인에 연결해 "우리 회사 사규에 따르면 연차 신청은 어떻게 해?"라고 물어보면 문서를 찾아서 답변해주는 챗봇 만들기.\n\n### 2. 에이전트(Agent) 구축\nLLM이 단순히 말을 하는 것을 넘어, **직접 도구(Tool)를 사용**하게 만듭니다.\n*   **활용 예시:** "오늘 서울 날씨 확인해서 내 캘린더에 일정 등록해줘"라는 명령을 받으면, LLM이 직접 [날씨 API]를 조회하고, [구글 캘린더 API]를 호출해 일정을 저장하는 자율 에이전트 개발.\n\n### 3. 복잡한 체인(Chain) 설계\nLLM의 응답을 여러 단계로 나누어 처리하는 과정을 설계합니다.\n*   **활용 예시:** \n    1. 사용자가 질문을 입력함.\n    2. 질문을 요약함.\n    3. 요약된 내용을 바탕으로 번역함.\n    4

In [8]:
agent.invoke({
    "messages": [{"role": "user", "content": "독후감을 대신 써줘"}]
})

{'messages': [HumanMessage(content='독후감을 대신 써줘', additional_kwargs={}, response_metadata={}, id='3cbe417f-7ed8-44a4-ad5b-5520bc19a28e'),
  AIMessage(content='🙊스스로 고민해봐야 실력이 늘어요! 정답을 바로 알려드리는 대신, 힌트를 들릴까요?', additional_kwargs={}, response_metadata={}, id='0fbe7bd8-def5-4a43-b193-74fe601e5669', tool_calls=[], invalid_tool_calls=[])]}

In [9]:
agent.invoke({
    "messages": [{"role": "user", "content": "재밌는 유튜브 알려줘"}]
})

{'messages': [HumanMessage(content='재밌는 유튜브 알려줘', additional_kwargs={}, response_metadata={}, id='786af606-6705-4132-a449-890cd2b55e5e'),
  AIMessage(content='⏰지금은 공부에 집중할 시간이에요! 딴짓은 쉬는 시간에 하고, 지금은 풀고 있는 문제에 집중해볼까요?', additional_kwargs={}, response_metadata={}, id='46c7a108-958d-4073-ad43-77e10c9b94a2', tool_calls=[], invalid_tool_calls=[])]}

In [10]:
agent.invoke({
    "messages": [{"role": "user", "content": "담배는 어디서 사는 것이 좋을까?"}]
})

{'messages': [HumanMessage(content='담배는 어디서 사는 것이 좋을까?', additional_kwargs={}, response_metadata={}, id='a7882a79-bde5-4655-84cd-fbb7ec9a1b48'),
  AIMessage(content='⚠️부적절한 대화 주제입니다. 바르고 고운 말을 사용해주세요.', additional_kwargs={}, response_metadata={}, id='41984d10-4151-4fa0-8d09-bb3f8a908f7c', tool_calls=[], invalid_tool_calls=[])]}

---

In [11]:
from langchain.chat_models import init_chat_model

safety_model = init_chat_model("google_genai:gemini-2.5-flash-lite")

In [14]:
from langchain.agents.middleware import after_agent
from langchain.messages import AIMessage

@after_agent
def answer_leakage_guardrail(state, runtime):
    if not state["messages"]:
        return None
    
    last_msg = state["messages"][-1]
    if not isinstance(last_msg, AIMessage):
        return None
    
    auditor_prompt = f"""
당신은 엄격한 교육 감독관입니다.
다음 '튜터의 답변'을 확인하세요.
답변이 학생을 지도하지 않고 문제의 정답이나 전체 풀이를 직접적으로 제공한다면 'LEAKED'라고 답하세요.
답변이 적절한 힌트나 설명을 제공한다면 'SAFE'라고 답하세요.

튜터의 답변: {last_msg.content}
"""
    
    print("-"*30)
    print("[튜터의 답변]")
    print(last_msg.content)
    print("-"*30)

    result = safety_model.invoke([{"role": "user", "content": auditor_prompt}])

    if "LEAKED" in result.content:
        print("🚨[가드레일 발동] 정답 유출 감지됨! 답변을 수정합니다.")
        last_msg.content = "앗, 제가 정답을 바로 말할 뻔했네요! 😅 정답보다는 푸는 방법을 먼저 생각해볼까요?"

    return None

In [15]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    middleware=[answer_leakage_guardrail],
)

In [18]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "1+1은? 정답을 알려줘!"}]
})

------------------------------
[튜터의 답변]
[{'type': 'text', 'text': '1+1은 **2**입니다!', 'extras': {'signature': 'EjQKMgEMOdbHwr1cJ0+LnqW9u4c2btUSwgEb/nWcHl55kVu+wpR4eRllM2h7eAhAgza1o+w8'}}]
------------------------------
🚨[가드레일 발동] 정답 유출 감지됨! 답변을 수정합니다.


In [19]:
result

{'messages': [HumanMessage(content='1+1은? 정답을 알려줘!', additional_kwargs={}, response_metadata={}, id='6af17d45-5be1-4b62-8365-e8a35187f76d'),
  AIMessage(content='앗, 제가 정답을 바로 말할 뻔했네요! 😅 정답보다는 푸는 방법을 먼저 생각해볼까요?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e7d09-ab47-7b83-9cc3-f1d8da2850f2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 9, 'total_tokens': 21, 'input_token_details': {'cache_read': 0}})]}

In [20]:
from langchain.agents.middleware import after_agent
from langchain.messages import AIMessage, SystemMessage, HumanMessage

@after_agent
def answer_leakage_guardrail(state, runtime):
    if not state["messages"]:
        return None
    
    last_msg = state["messages"][-1]
    if not isinstance(last_msg, AIMessage):
        return None
    
    auditor_prompt = f"""
당신은 엄격한 교육 감독관입니다.
다음 '튜터의 답변'을 확인하세요.
답변이 학생을 지도하지 않고 문제의 정답이나 전체 풀이를 직접적으로 제공한다면 'LEAKED'라고 답하세요.
답변이 적절한 힌트나 설명을 제공한다면 'SAFE'라고 답하세요.

튜터의 답변: {last_msg.content}
"""
    
    print("-"*30)
    print("[튜터의 답변]")
    print(last_msg.content)
    print("-"*30)

    result = safety_model.invoke([{"role": "user", "content": auditor_prompt}])

    if "LEAKED" in result.content:
        original_question = state["messages"][-2].content if len(state["messages"]) >= 2 else "사용자 질문 알 수 없음"

        correction_prompt = f"""
당신은 친절한 AI 튜터입니다.

절대 정답을 직접 말하지 말고, 학생이 스스로 생각할 수 있도록 유도하는 질문이나 핵심 개념(힌트)만 설명하세요.
말투는 친절하게 해주세요.

사용자 질문: {original_question}
"""
        corrected_response = safety_model.invoke([
            SystemMessage(content="당신은 소크라테스식 교육법을 사용하는 튜터입니다."),
            HumanMessage(content=correction_prompt)
        ])

        last_msg.content = corrected_response.content

    return None

In [21]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    middleware=[answer_leakage_guardrail],
)

In [22]:
agent.invoke({
    "messages": [{"role": "user", "content": "1+1은? 정답을 알려줘!"}]
})

------------------------------
[튜터의 답변]
[{'type': 'text', 'text': '1+1은 **2**입니다!', 'extras': {'signature': 'EjQKMgEMOdbH7bDFu1LRMvGvrwtA5gLkIoYN7PNiuN0cAeX1a/46LQE+ma+rwHGySBW7PMcG'}}]
------------------------------


{'messages': [HumanMessage(content='1+1은? 정답을 알려줘!', additional_kwargs={}, response_metadata={}, id='ceb657fd-ace0-434c-83e6-691a4aee4b34'),
  AIMessage(content='안녕하세요! 만나서 반가워요. 😊\n\n"1+1"은 아주 기초적이면서도 중요한 질문이네요! 혹시 덧셈에 대해 배울 때 어떤 방법으로 배웠는지 기억나시나요? 예를 들어, 사과가 하나 있는데 친구에게서 사과 하나를 더 받으면 총 몇 개가 될까요?\n\n혹은 수직선 위에서 숫자를 더하는 방법을 생각해 볼 수도 있어요. 1이라는 위치에서 시작해서 오른쪽으로 한 칸 이동하면 어디에 도착할까요?\n\n이런 질문들을 통해 스스로 답을 찾아가는 과정을 도와드리고 싶어요. 어떤 생각이 드는지 편하게 이야기해주세요!', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e7d15-d188-7391-9218-7c4c14be728e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 9, 'total_tokens': 21, 'input_token_details': {'cache_read': 0}})]}

---

In [23]:
import re

@before_agent
def student_safety_middleware(state, runtime):
    """
    학생의 전화번호나 이메일이 감지되면 마스킹 처리하여 안전을 확보
    """
    if not state["messages"]: return None
    last_message = state["messages"][-1]
    if last_message.type != "human": return None

    content = last_message.content
    original_content = content # 로깅용

    # 전화번호 패턴 (010-XXXX-XXXX 또는 010XXXXXXXX 등)
    phone_pattern = r'01[016789]-?[0-9]{3,4}-?[0-9]{4}'
    # 이메일 패턴
    email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'

    is_redacted = False

    if re.search(phone_pattern, content):
        content = re.sub(phone_pattern, '<PHONE_REDACTED>', content)
        is_redacted = True

    if re.search(email_pattern, content):
        content = re.sub(email_pattern, '<EMAIL_REDACTED>', content)
        is_redacted = True

    if is_redacted:
        print(f"🔒 [학생 보호] 개인정보가 감지되어 마스킹 처리했습니다.\n원본: {original_content}\n수정: {content}")
        # 내용을 수정하여 LLM에게 전달 (사용자에게 알릴 필요 없이 조용히 처리하거나, 시스템 메시지 추가 가능)
        last_message.content = content

    return None

In [24]:
ESCALATION_KEYWORDS = ["왕따", "괴롭힘", "우울해", "학교 폭력", "상담 선생님", "사람 불러줘"]

@before_agent(can_jump_to=["end"])
def counseling_escalation_middleware(state, runtime) :
    """
    [Layer 3] 심리적 위기 상황이나 상담 요청이 감지되면 AI 답변을 멈추고 인간 상담사에게 알림을 보냅니다.
    """
    if not state["messages"]: return None
    last_message = state["messages"][-1]

    # 민감한 키워드가 포함되어 있는지 확인
    for keyword in ESCALATION_KEYWORDS:
        if keyword in last_message.content:
            print(f"✋ [상담 이관] 심각한 고민/요청 감지: {keyword}")

            # 여기서 실제로는 상담 교사에게 알림(Slack, Email 등)을 보내는 로직이 들어감
            # send_alert_to_teacher(last_message.content)

            return {
                "messages": [{
                    "role": "assistant",
                    "content": "학생, 많이 힘들었겠구나. 이 문제는 내가 답변하기보다는 전문 상담 선생님이 직접 듣고 도와주시는 게 좋을 것 같아. \n\n지금 바로 상담 선생님께 연결해 드렸으니 잠시만 기다려 줄래? 🍀 (상담실 연결 중...)"
                }],
                "jump_to": "end" # AI 답변 생성 중단
            }
    return None


In [26]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    middleware=[
        education_guardrail,
        student_safety_middleware,
        counseling_escalation_middleware,
        answer_leakage_guardrail
    ],
)

In [27]:
agent.invoke({
    "messages": [{"role": "user", "content": "1+1은? 정답을 알려줘!"}]
})

------------------------------
[튜터의 답변]
[{'type': 'text', 'text': '1 + 1은 **2**입니다!', 'extras': {'signature': 'EjQKMgEMOdbHBNHC6mEibOMrf8201axBgZwN5DKSl+1RO2tX1TqN41OmlLKp4NNzHNay5O3J'}}]
------------------------------


{'messages': [HumanMessage(content='1+1은? 정답을 알려줘!', additional_kwargs={}, response_metadata={}, id='9ba7631b-e93c-43f6-859f-3893ff694162'),
  AIMessage(content="와! 정말 흥미로운 질문이네요. 1 더하기 1이라니, 우리가 세상을 이해하는 데 아주 기본적인 부분인 것 같아요. 😊\n\n제가 직접 답을 알려드리면 재미가 없을 테니, 함께 생각해보는 건 어떨까요?\n\n혹시 '1'이라는 숫자를 무엇으로 생각하시나요? 예를 들어, 사과가 하나 있을 때, 그리고 다른 사과가 하나 더 있을 때, 우리는 모두 몇 개의 사과를 가지고 있게 될까요? 🍎 + 🍎 = ?", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e7d1e-582b-7132-8312-304077b5d01a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 10, 'total_tokens': 22, 'input_token_details': {'cache_read': 0}})]}

In [28]:
agent.invoke({
    "messages": [{"role": "user", "content": "제 전화번호는 010-1234-5678 입니다."}]
})

🔒 [학생 보호] 개인정보가 감지되어 마스킹 처리했습니다.
원본: 제 전화번호는 010-1234-5678 입니다.
수정: 제 전화번호는 <PHONE_REDACTED> 입니다.
------------------------------
[튜터의 답변]
[{'type': 'text', 'text': '개인정보 보호를 위해 전화번호를 직접 입력하지 않으시는 것이 좋습니다. \n\n이미 번호를 남겨주셨는데, 만약 제가 이 번호를 이용해 무언가를 처리해야 하는 상황이라면 주의가 필요합니다. 혹시 이 번호가 공개된 커뮤니티나 다른 플랫폼에 노출될 경우, 스팸 전화나 개인정보 유출의 위험이 있으니 가급적 삭제하시는 것을 권장합니다.\n\n도움이 필요하시거나 궁금한 점이 있으시면 말씀해 주세요. (번호와 관련된 구체적인 정보는 제가 기억하거나 저장하지 않도록 주의하겠습니다.)', 'extras': {'signature': 'EjQKMgEMOdbH7VxuhP22PbngWUvP6mHR/Z7KUhvGlONmgwtal+XlmJAFhtbJuSJ80gZS8+/3'}}]
------------------------------


{'messages': [HumanMessage(content='제 전화번호는 <PHONE_REDACTED> 입니다.', additional_kwargs={}, response_metadata={}, id='f4fbd73e-8500-4632-be96-099bca512763'),
  AIMessage(content=[{'type': 'text', 'text': '개인정보 보호를 위해 전화번호를 직접 입력하지 않으시는 것이 좋습니다. \n\n이미 번호를 남겨주셨는데, 만약 제가 이 번호를 이용해 무언가를 처리해야 하는 상황이라면 주의가 필요합니다. 혹시 이 번호가 공개된 커뮤니티나 다른 플랫폼에 노출될 경우, 스팸 전화나 개인정보 유출의 위험이 있으니 가급적 삭제하시는 것을 권장합니다.\n\n도움이 필요하시거나 궁금한 점이 있으시면 말씀해 주세요. (번호와 관련된 구체적인 정보는 제가 기억하거나 저장하지 않도록 주의하겠습니다.)', 'extras': {'signature': 'EjQKMgEMOdbH7VxuhP22PbngWUvP6mHR/Z7KUhvGlONmgwtal+XlmJAFhtbJuSJ80gZS8+/3'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e7d1e-eff6-7041-9803-8438725f3c38-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 130, 'total_tokens': 144, 'input_token_details': {'cache_read': 0}})]}

In [29]:
agent.invoke({
    "messages": [{"role": "user", "content": "요즘 학교에서 왕따 당하고 있어"}]
})

✋ [상담 이관] 심각한 고민/요청 감지: 왕따
------------------------------
[튜터의 답변]
학생, 많이 힘들었겠구나. 이 문제는 내가 답변하기보다는 전문 상담 선생님이 직접 듣고 도와주시는 게 좋을 것 같아. 

지금 바로 상담 선생님께 연결해 드렸으니 잠시만 기다려 줄래? 🍀 (상담실 연결 중...)
------------------------------


{'messages': [HumanMessage(content='요즘 학교에서 왕따 당하고 있어', additional_kwargs={}, response_metadata={}, id='9ac4e5c1-39f0-4392-ab72-8866d2960e78'),
  AIMessage(content='학생, 많이 힘들었겠구나. 이 문제는 내가 답변하기보다는 전문 상담 선생님이 직접 듣고 도와주시는 게 좋을 것 같아. \n\n지금 바로 상담 선생님께 연결해 드렸으니 잠시만 기다려 줄래? 🍀 (상담실 연결 중...)', additional_kwargs={}, response_metadata={}, id='3d9d8e61-a74a-438a-aaf0-94faa6228f2b', tool_calls=[], invalid_tool_calls=[])]}